In [19]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import FigueredoSolver
from signalClass import *
import time

In [20]:
np.random.seed(24102000)
n = 128

construct blur matrix

In [21]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [22]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 15)
RndSignal.generate_GG_realization(0, sigma, 2)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [23]:
mu = 0.5
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [24]:
#begin solver construction
np.random.seed(24102001)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = FigueredoSolver.FigueredoSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [25]:
iters = 800

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [26]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 800
2 / 800
3 / 800
4 / 800
5 / 800
6 / 800
7 / 800
8 / 800
9 / 800
10 / 800
11 / 800
12 / 800
13 / 800
14 / 800
15 / 800
16 / 800
17 / 800
18 / 800
19 / 800
20 / 800
21 / 800
22 / 800
23 / 800
24 / 800
25 / 800
26 / 800
27 / 800
28 / 800
29 / 800
30 / 800
31 / 800
32 / 800
33 / 800
34 / 800
35 / 800
36 / 800
37 / 800
38 / 800
39 / 800
40 / 800
41 / 800
42 / 800
43 / 800
44 / 800
45 / 800
46 / 800
47 / 800
48 / 800
49 / 800
50 / 800
51 / 800
52 / 800
53 / 800
54 / 800
55 / 800
56 / 800
57 / 800
58 / 800
59 / 800
60 / 800
61 / 800
62 / 800
63 / 800
64 / 800
65 / 800
66 / 800
67 / 800
68 / 800
69 / 800
70 / 800
71 / 800
72 / 800
73 / 800
74 / 800
75 / 800
76 / 800
77 / 800
78 / 800
79 / 800
80 / 800
81 / 800
82 / 800
83 / 800
84 / 800
85 / 800
86 / 800
87 / 800
88 / 800
89 / 800
90 / 800
91 / 800
92 / 800
93 / 800
94 / 800
95 / 800
96 / 800
97 / 800
98 / 800
99 / 800
100 / 800
101 / 800
102 / 800
103 / 800
104 / 800
105 / 800
106 / 800
107 / 800
108 / 800
109 / 800
110 / 800
111 / 80

In [27]:
np.savez_compressed(
    "./FigueredoADMMTVL1-Gauss.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)